# Update Canada's Trade Relationships from Google Drive

## Purpose
Updates Canada's trade relationship data by loading import and export statistics from Google Sheets, cleaning the data, and calculating country-specific trade percentages used in the risk index calculation.

## Data Source
* **Source**: StatsCan trade data (February 2021 - February 2026)
* **Google Sheets**: Import and export data for Canada
  * Imports: Sheet tab gid=0
  * Exports: Sheet tab gid=1809920598

## Workflow
1. Load import and export data from Google Sheets into Spark DataFrames
2. Clean column names (remove special characters, standardize formatting)
3. Aggregate trade quantities by country across all years
4. Filter out Canada from the datasets
5. Combine China and Hong Kong records into a single "China" entry
6. Calculate trade percentages:
   * `total_import_percent`: Each country's share of Canada's total imports
   * `total_export_percent`: Each country's share of Canada's total exports

## Output
* `import_agg`: Country-level import quantities and percentages
* `export_agg`: Country-level export quantities and percentages

These percentages are used as the **Exposure (E)** component in the risk index formula: `Index = 100 × Shock × Exposure`

In [0]:
import pandas as pd
import numpy as np


In [0]:
# Data was obtained from StatsCan using their webapp to download import and export data for canada for the last 5 years (Feb 2021 - Feb 2026)
# https://www150.statcan.gc.ca/n1/pub/71-607-x/71-607-x2021004-eng.htm
# Imports https://docs.google.com/spreadsheets/d/1LYwuRewKYRpudNzhzupYvEyNScuHxG3amPnpgJC5z6Y/edit?gid=0#gid=0
# Exports https://docs.google.com/spreadsheets/d/1LYwuRewKYRpudNzhzupYvEyNScuHxG3amPnpgJC5z6Y/edit?gid=1809920598#gid=1809920598

sheet_id = "1LYwuRewKYRpudNzhzupYvEyNScuHxG3amPnpgJC5z6Y"
import_gid = "0" # sheet tab ID
export_gid = "1809920598"
import_url = f"https://docs.google.com/spreadsheets/d/{sheet_id}/export?format=csv&gid={import_gid}"
export_url = f"https://docs.google.com/spreadsheets/d/{sheet_id}/export?format=csv&gid={export_gid}"

import_df = spark.createDataFrame(pd.read_csv(import_url))
export_df = spark.createDataFrame(pd.read_csv(export_url))

# Clean column names 
def clean_column_names(df):
    for col in df.columns:
        clean_col = col.replace('(', '').replace(')', '').replace(',', '').replace(' ', '_').replace('/','per')
        if col != clean_col:
            df = df.withColumnRenamed(col, clean_col)
    return df

import_df = clean_column_names(import_df)
export_df = clean_column_names(export_df)



In [0]:
#Now aggregate the data for country (all years combined)
from pyspark.sql.functions import sum as spark_sum

# Aggregate imports by country (sum across all years)
import_agg = import_df \
    .groupBy("Country") \
    .agg(spark_sum("Quantity").alias("Total_Import_Quantity")) \
    .orderBy("Country")


# Aggregate exports by country (sum across all years)
export_agg = export_df \
    .groupBy("Country") \
    .agg(spark_sum("Quantity").alias("Total_Export_Quantity")) \
    .orderBy("Country")



In [0]:
from pyspark.sql.functions import when, sum as spark_sum

# Ensure Canada is not in either dataframe
import_agg = import_agg.filter(~(import_agg.Country == "Canada"))
export_agg = export_agg.filter(~(export_agg.Country == "Canada"))

#china and hong kong need to be combined
export_agg = export_agg.withColumn("Country", 
    when(export_agg.Country == "Hong Kong", "China")
    .when(export_agg.Country == "Hong Kong, China", "China")
    .otherwise(export_agg.Country)
)
import_agg = import_agg.withColumn("Country", 
    when(import_agg.Country == "Hong Kong", "China")
    .when(import_agg.Country == "Hong Kong, China", "China")
    .otherwise(import_agg.Country))

# Re-aggregate after renaming to combine duplicate country rows (e.g., China + Hong Kong -> China)
export_agg = export_agg.groupBy("Country").agg(spark_sum("Total_Export_Quantity").alias("Total_Export_Quantity")).orderBy("Country")
import_agg = import_agg.groupBy("Country").agg(spark_sum("Total_Import_Quantity").alias("Total_Import_Quantity")).orderBy("Country")

In [0]:
# Now calculate total_import_percent and total_export_percent
from pyspark.sql.functions import sum as spark_sum, col

total_import = import_agg.select(spark_sum("Total_Import_Quantity")).collect()[0][0]
total_export = export_agg.select(spark_sum("Total_Export_Quantity")).collect()[0][0]

import_agg = import_agg.withColumn("total_import_percent", col("Total_Import_Quantity") / total_import)
export_agg = export_agg.withColumn("total_export_percent", col("Total_Export_Quantity") / total_export)


Country,Total_Import_Quantity,total_import_percent
Colombia,1523245,0.17052886568069145
Côte d'Ivoire,214945,0.02406331682279359
Ecuador,2344655,0.26248657147245624
Germany,3,3.358531273971517E-7
India,1,1.1195104246571723E-7
Niger,2,2.2390208493143446E-7
Norway,199631,0.022348898558473596
Oman,1,1.1195104246571723E-7
Spain,0,0.0
Türkiye,0,0.0


Country,Total_Export_Quantity,total_export_percent
Brazil,100,1.0644308203615061E-7
China,29056310,0.030928431889978233
Colombia,15,1.596646230542259E-8
Dominican Republic,0,0.0
Finland,101752,1.0830796483342398E-4
Germany,2585983,0.0027526000061309087
India,262500,2.7941309034489536E-4
Italy,202035,2.150522807917369E-4
"Korea, South",1482692,0.0015782230619034421
Mexico,2,2.1288616407230124E-9
